# 14x Verify CityFlow 14e Siblings
CPU-only verifier for the 14g, 14h, 14i, 14j, and 14k CityFlow sibling headlines. It reuses frozen Stage-1/Stage-2 artifacts and runs only stages 3, 4, and 5.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/MRKDaGods/gp.git"
EXPECTED_MASTER_SHA_AT_BUILD = "7c1120305ef51b341e292459fd85d30703480bca"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
RESULTS_PATH = WORK_DIR / "14x_verify_results.json"

def run(cmd, *, cwd=None, check=True, capture=False):
    print("$", " ".join(map(str, cmd)))
    if capture:
        return subprocess.check_output(list(map(str, cmd)), cwd=cwd, text=True)
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check)

def pip_install(*args):
    run([sys.executable, "-m", "pip", "install", "-q", *args])

if not PROJECT.exists():
    run(["git", "clone", "--branch", "master", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    run(["git", "-C", PROJECT, "fetch", "origin", "master", "--depth", "1"])
    run(["git", "-C", PROJECT, "checkout", "master"])
    run(["git", "-C", PROJECT, "reset", "--hard", "origin/master"])

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
head_sha = run(["git", "rev-parse", "HEAD"], capture=True).strip()
print("resolved git SHA:", head_sha)
print("expected master SHA at notebook build:", EXPECTED_MASTER_SHA_AT_BUILD)
if head_sha != EXPECTED_MASTER_SHA_AT_BUILD:
    print(f"WARN: master moved since notebook build (expected {EXPECTED_MASTER_SHA_AT_BUILD}, got {head_sha}); proceeding because metric assertions are authoritative.")
print("python:", sys.version)
print("platform:", platform.platform())

pip_install("faiss-cpu", "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip_install("filterpy", "ftfy", "lapx", "timm")
pip_install("--no-deps", "ultralytics")
pip_install("--no-deps", "boxmot==11.0.3")
pip_install("--no-deps", "-e", ".")

help_text = run([sys.executable, "scripts/run_pipeline.py", "--help"], capture=True)
RUN_ID_SHIM_APPLIED = False
if "--run-id" not in help_text:
    script_path = Path("scripts/run_pipeline.py")
    script_text = script_path.read_text(encoding="utf-8")
    script_text = script_text.replace(
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n',
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n'
        '@click.option("--run-id", default=None, help="Run id/name for the output directory")\n',
    )
    script_text = script_text.replace(
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, override: tuple):',
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, run_id: str | None, override: tuple):',
    )
    script_text = script_text.replace(
        '    if apply_cpu_when_no_cuda(cfg):\n',
        '    if run_id:\n        cfg.project.run_name = run_id\n\n    if apply_cpu_when_no_cuda(cfg):\n',
    )
    script_path.write_text(script_text, encoding="utf-8")
    RUN_ID_SHIM_APPLIED = True
    print("Applied runtime-only --run-id compatibility shim")
else:
    print("run_pipeline.py already exposes --run-id")

for mount in [Path("/tmp"), WORK_DIR]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount}: {free / 1024**3:.1f} GiB free / {total / 1024**3:.1f} GiB total")

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import tarfile
import time

INPUT_ROOT = Path("/kaggle/input")

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    print(f"wrote {path}")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def symlink_or_copy(src: Path, dst: Path) -> None:
    if dst.exists() or dst.is_symlink():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

def list_input_roots(max_items: int = 80) -> list[str]:
    roots = sorted(str(path) for path in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for item in roots[:max_items]:
        print(item)
    if len(roots) > max_items:
        print(f"... {len(roots) - max_items} more")
    return roots

def find_input_dir(owner_slug: str, *hints: str) -> Path | None:
    owner, slug = owner_slug.split("/", 1)
    direct_candidates = [
        INPUT_ROOT / slug,
        INPUT_ROOT / owner_slug.replace("/", "-"),
        INPUT_ROOT / "notebooks" / owner / slug,
        INPUT_ROOT / "datasets" / owner / slug,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate
    wanted = [slug.lower(), *(hint.lower() for hint in hints)]
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir():
            text = str(path).lower()
            if all(token in text for token in wanted if token):
                return path
    return None

def find_file(names: list[str], *, owner_slug: str | None = None, hints: tuple[str, ...] = ()) -> Path:
    roots = []
    if owner_slug:
        root = find_input_dir(owner_slug, *hints)
        if root is not None:
            roots.append(root)
    roots.append(INPUT_ROOT)
    matches = []
    lowered_hints = tuple(h.lower() for h in hints)
    for root in roots:
        if root is None or not root.exists():
            continue
        for name in names:
            matches.extend(root.rglob(name))
    if owner_slug or hints:
        filtered = []
        for path in matches:
            text = str(path).lower()
            if owner_slug and owner_slug.split("/", 1)[1].lower() in text:
                filtered.append(path)
            elif lowered_hints and all(h in text for h in lowered_hints):
                filtered.append(path)
        matches = filtered or matches
    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    if not matches:
        raise FileNotFoundError(f"Could not find {names} owner_slug={owner_slug} hints={hints}")
    print("selected", matches[0])
    return matches[0]

def find_wildtrack_root() -> Path:
    markers = {"Image_subsets", "annotations_positions", "calibrations"}
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir():
            children = {child.name for child in path.iterdir() if child.is_dir()}
            if markers.issubset(children):
                print("WILDTRACK root:", path)
                return path
    raise FileNotFoundError("Could not find WILDTRACK root with Image_subsets, annotations_positions, calibrations")

def find_cityflow_root() -> Path:
    candidates = [
        INPUT_ROOT / "data-aicity-2023-track-2",
        INPUT_ROOT / "datasets" / "thanhnguyenle" / "data-aicity-2023-track-2",
    ]
    for root in candidates:
        if root.exists():
            return root
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir() and ((path / "train").exists() or (path / "AIC22_Track1_MTMC_Tracking").exists()):
            return path
    raise FileNotFoundError("Could not find CityFlowV2/AIC22 dataset root")

def prepare_cityflow_layout(source_root: Path) -> Path:
    raw_parent = PROJECT / "data" / "raw"
    tmp_parent = Path("/tmp/datasets")
    tmp_parent.mkdir(parents=True, exist_ok=True)
    if raw_parent.exists() or raw_parent.is_symlink():
        if raw_parent.is_symlink() or raw_parent.is_file():
            raw_parent.unlink()
        else:
            shutil.rmtree(raw_parent)
    raw_parent.parent.mkdir(parents=True, exist_ok=True)
    raw_parent.symlink_to(tmp_parent, target_is_directory=True)
    data_raw = tmp_parent / "cityflowv2"
    data_raw.mkdir(parents=True, exist_ok=True)
    roots = []
    if (source_root / "AIC22_Track1_MTMC_Tracking").exists():
        roots.append(source_root / "AIC22_Track1_MTMC_Tracking")
    roots.append(source_root)
    for root in roots:
        for split in ["train", "validation", "test"]:
            split_dir = root / split
            if not split_dir.exists():
                continue
            for scene_dir in sorted(split_dir.iterdir()):
                if not scene_dir.is_dir():
                    continue
                for cam_dir in sorted(scene_dir.iterdir()):
                    if not cam_dir.is_dir():
                        continue
                    flat = data_raw / f"{scene_dir.name}_{cam_dir.name}"
                    if not flat.exists():
                        symlink_or_copy(cam_dir, flat)
    expected = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]
    missing = [cam for cam in expected if not (data_raw / cam / "gt" / "gt.txt").exists()]
    assert not missing, f"CityFlow GT camera layout missing: {missing} under {data_raw}"
    print("CityFlow layout ready:", data_raw)
    return data_raw

In [ ]:
import numpy as np

list_input_roots()
CITYFLOW_ROOT = prepare_cityflow_layout(find_cityflow_root())
GT_ROOT = str(CITYFLOW_ROOT)

def extract_checkpoint_for(owner_slug: str, work_name: str, hints: tuple[str, ...]) -> Path:
    checkpoint = find_file(["checkpoint.tar.gz"], owner_slug=owner_slug, hints=hints)
    out_dir = Path("/tmp/14x_sources") / work_name
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(checkpoint, "r:gz") as tar:
        tar.extractall(out_dir)
    meta = out_dir / "run_metadata.json"
    if meta.exists():
        run_name = json.loads(meta.read_text(encoding="utf-8"))["run_name"]
        return out_dir / run_name
    candidates = [p for p in out_dir.rglob("stage2") if p.is_dir()]
    assert candidates, f"no stage2 in {out_dir}"
    return candidates[0].parent

SOURCES = {
    "14g": extract_checkpoint_for("yahiaakhalafallah/14g-dinov2-4view-tta-stage2", "14g", ("14g", "dinov2")),
    "14h": extract_checkpoint_for("yahiaakhalafallah/14h-robust-tracklet-pooling", "14h", ("14h", "robust")),
}
Q_ROOT = find_input_dir("yahiaakhalafallah/14j-r50-ibn-features", "14j", "r50")
assert Q_ROOT is not None, "14j R50-IBN feature mount missing"
print("quaternary source:", Q_ROOT)
for name, run_dir in SOURCES.items():
    for rel in ["stage1", "stage2/embeddings.npy", "stage2/embeddings_tertiary.npy", "stage2/hsv_features.npy", "stage2/embedding_index.json"]:
        assert (run_dir / rel).exists(), f"{name} missing {rel}: {run_dir}"
    print(name, run_dir)

In [ ]:
from scripts.filter_tracklets import filter_stage_outputs

def stage_run_dir(label: str) -> Path:
    return PROJECT / "data" / "outputs" / f"run_verify_{label}"

def materialize_source(label: str, source: Path) -> Path:
    run_dir = stage_run_dir(label)
    run_dir.mkdir(parents=True, exist_ok=True)
    symlink_or_copy(source / "stage1", run_dir / "stage1")
    symlink_or_copy(source / "stage2", run_dir / "stage2")
    return run_dir

def parse_metrics(report_path: Path) -> dict:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    details = report.get("details", {}) or {}
    return {
        "mtmc_idf1": report.get("mtmc_idf1", details.get("mtmc_idf1", report.get("idf1", report.get("IDF1")))),
        "id_switches": report.get("id_switches", details.get("id_switches", report.get("IDS"))),
        "raw": report,
    }

def normalize_quaternary(base_stage2: Path) -> Path:
    base_index = json.loads((base_stage2 / "embedding_index.json").read_text(encoding="utf-8"))
    q_index_path = find_file(["embedding_index.json"], owner_slug="yahiaakhalafallah/14j-r50-ibn-features", hints=("14j", "r50"))
    q_index = json.loads(q_index_path.read_text(encoding="utf-8"))
    assert base_index == q_index, "R50-IBN embedding_index.json does not match 14h stage2 ordering"
    q_path = find_file(["embeddings_quaternary_l2.npy", "embeddings_quaternary.npy", "embeddings.npy"], owner_slug="yahiaakhalafallah/14j-r50-ibn-features", hints=("14j", "r50"))
    arr = np.load(q_path)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    arr = arr / np.maximum(norms, 1e-12)
    out = Path("/kaggle/working/14x_sources/embeddings_quaternary_l2.npy")
    out.parent.mkdir(parents=True, exist_ok=True)
    np.save(out, arr.astype("float32"))
    assert arr.shape[0] == len(base_index)
    print("normalized quaternary:", out, arr.shape)
    return out

QUATERNARY_PATH = normalize_quaternary(SOURCES["14h"] / "stage2")

In [ ]:
TARGETS = [
    {"label": "14g_s0", "source": "14g", "idf1_target": 0.77902, "idf1_tol": 0.005, "id_sw_target": 154, "id_sw_assert": "informational", "tertiary_weight": 0.525, "quaternary_weight": 0.0, "thr": 0.48, "aqe_k": 2, "fic": 0.5, "filter": None},
    {"label": "14h_m0", "source": "14h", "idf1_target": 0.77936, "idf1_tol": 0.005, "id_sw_target": 154, "id_sw_assert": "exact", "tertiary_weight": 0.525, "quaternary_weight": 0.0, "thr": 0.48, "aqe_k": 2, "fic": 0.5, "filter": None},
    {"label": "14i_f0", "source": "14h", "idf1_target": 0.77936, "idf1_tol": 0.005, "id_sw_target": 154, "id_sw_assert": "exact", "tertiary_weight": 0.525, "quaternary_weight": 0.0, "thr": 0.48, "aqe_k": 2, "fic": 0.5, "filter": {"min_length": 0, "min_avg_confidence": 0.0}},
    {"label": "14i_f2", "source": "14h", "idf1_target": 0.77964, "idf1_tol": 0.005, "id_sw_target": 120, "id_sw_assert": "informational", "tertiary_weight": 0.525, "quaternary_weight": 0.0, "thr": 0.48, "aqe_k": 2, "fic": 0.5, "filter": {"min_length": 3, "min_avg_confidence": 0.35}},
    {"label": "14j_w14", "source": "14h", "idf1_target": 0.78032, "idf1_tol": 0.005, "id_sw_target": 207, "id_sw_assert": "informational", "tertiary_weight": 0.525, "quaternary_weight": 0.30, "thr": 0.48, "aqe_k": 2, "fic": 0.5, "filter": None},
    {"label": "14k_k7", "source": "14h", "idf1_target": 0.78079, "idf1_tol": 0.005, "id_sw_target": 213, "id_sw_assert": "informational", "tertiary_weight": 0.45, "quaternary_weight": 0.45, "thr": 0.46, "aqe_k": 2, "fic": 0.5, "filter": None},
]
print(json.dumps(TARGETS, indent=2))

In [ ]:
results = []

def run_target(row: dict) -> dict:
    label = row["label"]
    source = SOURCES[row["source"]]
    run_dir = stage_run_dir(label)
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    if row["filter"]:
        summary_path = run_dir / "filter_summary.json"
        filter_stage_outputs(
            stage1_dir=source / "stage1",
            stage2_dir=source / "stage2",
            output_stage1_dir=run_dir / "stage1",
            output_stage2_dir=run_dir / "stage2",
            summary_path=summary_path,
            min_length=row["filter"]["min_length"],
            min_avg_confidence=row["filter"]["min_avg_confidence"],
        )
    else:
        materialize_source(label, source)
    overrides = [
        f"project.run_name=run_verify_{label}",
        "project.output_dir=/kaggle/working/gp/data/outputs",
        "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
        f"stage4.association.tertiary_embeddings.path={run_dir / 'stage2' / 'embeddings_tertiary.npy'}",
        f"stage4.association.tertiary_embeddings.weight={row['tertiary_weight']}",
        f"stage4.association.quaternary_embeddings.path={QUATERNARY_PATH}",
        f"stage4.association.quaternary_embeddings.weight={row['quaternary_weight']}",
        f"stage4.association.graph.similarity_threshold={row['thr']}",
        f"stage4.association.query_expansion.k={row['aqe_k']}",
        f"stage4.association.fic.regularisation={row['fic']}",
        f"stage5.ground_truth_dir={GT_ROOT}",
    ]
    cmd = [sys.executable, "scripts/run_pipeline.py", "--config", "configs/datasets/cityflowv2.yaml", "--stages", "3,4,5", "--run-id", f"run_verify_{label}"]
    for override in overrides:
        cmd.extend(["--override", override])
    started = time.time()
    run(cmd)
    parsed = parse_metrics(run_dir / "stage5" / "evaluation_report.json")
    got = float(parsed["mtmc_idf1"])
    idsw = parsed["id_switches"]
    status = "PASS" if abs(got - row["idf1_target"]) <= row["idf1_tol"] else "FAIL"
    if row["id_sw_assert"] == "exact" and int(idsw) != int(row["id_sw_target"]):
        status = "FAIL"
    return {**row, "observed_idf1": got, "observed_id_switches": idsw, "delta": got - row["idf1_target"], "status": status, "elapsed_sec": time.time() - started}

for row in TARGETS:
    try:
        out = run_target(row)
    except Exception as exc:
        out = {**row, "status": "ERROR", "error": repr(exc)}
    results.append(out)
    write_json(RESULTS_PATH, {"verifier": "14x_verify_cityflow_siblings", "repo_sha": head_sha, "run_id_shim_applied": RUN_ID_SHIM_APPLIED, "results": results, "all_passed": all(r.get("status") == "PASS" for r in results)})
    print(json.dumps(out, indent=2))
failures = [r for r in results if r.get("status") != "PASS"]
print("label observed target delta idsw status")
for r in results:
    print(r["label"], r.get("observed_idf1"), r["idf1_target"], r.get("delta"), r.get("observed_id_switches"), r["status"])
assert not failures, json.dumps(failures, indent=2)